# 01 · Exploratory analysis

Questions to answer before modeling: how rare is fraud, where is it concentrated, what does the identity
LEFT join change, how do amounts and timing differ, and what does missingness tell us?

All heavy lifting is SQL against the DuckDB file built by `make features`; this notebook only displays
results. Set `FRAUD_PROFILE=sample` to run it on synthetic data.

In [ ]:
import os
import altair as alt
import numpy as np
import pandas as pd

from fraud import charts, db  # charts registers the project's Altair theme
from fraud.config import get_paths
from fraud.io import load_json

PROFILE = os.environ.get("FRAUD_PROFILE", "full")  # "sample" = synthetic data
paths = get_paths(PROFILE)
pd.set_option("display.max_columns", 50, "display.width", 160)
print("profile:", PROFILE)

In [ ]:
conn = db.connect(paths.db, read_only=True)
q = lambda sql, **kw: conn.execute(db.read_sql(sql, **kw) if sql.endswith(".sql") else sql).df()
support = 500 if PROFILE == "full" else 30

## 1. Class balance and where fraud concentrates

In [ ]:
overall = q("SELECT count(*) AS n, avg(isFraud) AS fraud_rate, "
            "sum(TransactionAmt) FILTER (WHERE isFraud = 1) / sum(TransactionAmt) AS fraud_usd_share FROM features")
overall

In [ ]:
cohorts = q("analysis/01_fraud_rate_by_cohort.sql", min_support=support)
cohorts

In [ ]:
base_rate = float(overall.fraud_rate[0])
bars = alt.Chart(cohorts).mark_bar(color=charts.SERIES[0], cornerRadiusEnd=4, height=14).encode(
    y=alt.Y("value:N", title=None, sort="-x"), x=alt.X("fraud_rate:Q", axis=alt.Axis(format="%"), title="Fraud rate"),
    tooltip=["dimension", "value", "transactions", alt.Tooltip("fraud_rate:Q", format=".2%"), "lift"])
rule = alt.Chart(pd.DataFrame({"x": [base_rate]})).mark_rule(strokeDash=[4, 4], color=charts.MUTED).encode(x="x:Q")
alt.layer(bars, rule).properties(width=320, height=alt.Step(18)).facet(
    row=alt.Row("dimension:N", title=None), data=cohorts).resolve_scale(y="independent").properties(
    title=alt.Title("Fraud rate by cohort", subtitle="dashed = overall fraud rate"))

## 2. What an INNER join to identity would have done

In [ ]:
q("analysis/02_identity_missingness.sql")

**Finding (DRAFT, rewrite in your voice after running):** _N% of transactions have no identity row. Those
rows carry X% of fraud dollars at a fraud rate of Y% vs Z% with identity. An INNER join would have thrown
them away and changed the base rate the model learns._

## 3. Amounts: fraud vs legitimate

In [ ]:
amt = q("SELECT isFraud, TransactionAmt, amt_cents FROM features USING SAMPLE 200000 ROWS (reservoir, 0)")
amt["log_amount"] = np.log10(amt.TransactionAmt)
amt["class"] = np.where(amt.isFraud == 1, "fraud", "legitimate")
alt.Chart(amt).transform_bin("bin", "log_amount", bin=alt.Bin(maxbins=60)).transform_aggregate(
    n="count()", groupby=["bin", "class"]).transform_joinaggregate(total="sum(n)", groupby=["class"]
).transform_calculate(share="datum.n / datum.total").mark_line(interpolate="step-after").encode(
    x=alt.X("bin:Q", title="log10(amount, USD)"), y=alt.Y("share:Q", title="Share of class", axis=alt.Axis(format="%")),
    color=alt.Color("class:N", scale=alt.Scale(domain=["legitimate", "fraud"], range=charts.SERIES[:2]))
).properties(width=560, height=260, title="Amount distribution by class (each class sums to 100%)")

In [ ]:
amt["cents_band"] = pd.cut(amt.amt_cents, [-0.001, 0.0005, 0.5, 0.9495, 1], labels=["$.00", "$.01-.50", "$.51-.95", "$.95-.99"])
amt.groupby("cents_band", observed=True).agg(n=("isFraud", "size"), fraud_rate=("isFraud", "mean"))

## 4. Timing (relative hour; the reference date is unknown)

In [ ]:
hourly = q("SELECT hour, count(*) AS n, avg(isFraud) AS fraud_rate FROM features GROUP BY hour ORDER BY hour")
alt.Chart(hourly).mark_line(point=True, color=charts.SERIES[0]).encode(
    x=alt.X("hour:O", title="Hour of day (relative to unknown reference)"),
    y=alt.Y("fraud_rate:Q", axis=alt.Axis(format="%"), title="Fraud rate"),
    tooltip=["hour", "n", alt.Tooltip("fraud_rate:Q", format=".2%")]).properties(width=560, height=240)

_Hour 0 is not midnight: the time axis has no calendar anchor, so only relative patterns count._

## 5. Missingness as a signal

In [ ]:
cols = [c for c in conn.execute("DESCRIBE base").df().column_name if c not in ("TransactionID", "isFraud")]
exprs = ", ".join(f"avg(CAST({c} IS NULL AS DOUBLE)) AS \"{c}\"" for c in cols)
miss = conn.execute(f"SELECT {exprs} FROM base").df().T.rename(columns={0: "missing_share"})
print("columns >90% missing:", int((miss.missing_share > 0.9).sum()), "of", len(miss))
probe = [c for c in miss.index if 0.05 < miss.missing_share[c] < 0.95]
rows = []
for c in probe:
    r = conn.execute(f"SELECT avg(isFraud) FILTER (WHERE {c} IS NULL), avg(isFraud) FILTER (WHERE {c} IS NOT NULL) FROM base").fetchone()
    rows.append({"column": c, "missing_share": miss.missing_share[c], "fraud_if_missing": r[0], "fraud_if_present": r[1]})
mt = pd.DataFrame(rows)
mt["ratio"] = mt.fraud_if_present / mt.fraud_if_missing
mt.sort_values("ratio", ascending=False).head(15)

## 6. Drift across the time split

In [ ]:
pd.DataFrame(load_json(paths.reports / "train_results.json")["splits"])

**Findings (DRAFT, rewrite after running):**
- _Base rate and how it drifts across splits_
- _Highest-risk cohorts (product, device, identity) and their lift_
- _Amount / cents pattern_
- _Which missingness flags carry signal_